In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os
import time
from PIL import Image

# ==========================================
# 1. Hardware & Configuration
# ==========================================
# Detect AMD GPU (ROCm/DirectML) or fallback to CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ryzen 7 7800X3D has 8 cores / 16 threads. 
# We use 8 workers to parallelize the heavy data augmentation we are about to do.
NUM_WORKERS = 8 
BATCH_SIZE = 128  # Efficient for 32GB RAM
IMG_SIZE = 100
EPOCHS = 15
NUM_CLASSES = 33

print(f"🚀 Project Running on: {DEVICE}")
print(f"⚙️  Config: {NUM_WORKERS} Workers | Batch Size {BATCH_SIZE}")

# ==========================================
# 2. Data Preparation: The "Robustness" Tweak
# ==========================================
TRAIN_DIR = "Fruits/train/train"  
TEST_DIR = "Fruits/test/test"

# TWEAK 1: Aggressive Augmentation
# This aligns with the "Data Drift" concerns in Exercise 8d.
# We force the model to see "bad" versions of the fruit so it learns robust features.
robust_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(30),          # Rotate up to 30 degrees
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)), # Zoom/Shift
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), # Lighting changes
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Validation/Test should remain "clean" (No random augmentation)
clean_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load Data
try:
    full_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=robust_transforms)
except FileNotFoundError:
    print(f"❌ Error: Could not find {TRAIN_DIR}. Please check paths.")
    raise

# TWEAK 2: Strict Data Splitting (Exercise 7a)
# 80% Training / 20% Validation
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

# IMPORTANT: Validation set should not have augmentation.
# We override the transform for the validation subset.
val_ds.dataset.transform = clean_transforms

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, 
                        num_workers=NUM_WORKERS, pin_memory=True)

class_names = full_dataset.classes
print(f"✅ Data Prepared: {len(train_ds)} Train | {len(val_ds)} Val")

# ==========================================
# 3. Model Architecture (Exercise 8a)
# ==========================================
class FruitCNN(nn.Module):
    def __init__(self, use_dropout=False, use_batchnorm=False):
        super(FruitCNN, self).__init__()
        
        self.use_dropout = use_dropout
        
        # Feature Extractor
        # We use a modular block design: Conv -> BN -> ReLU -> MaxPool
        layers = []
        
        # Block 1: 3 -> 32
        layers.append(nn.Conv2d(3, 32, kernel_size=3, padding=1))
        if use_batchnorm: layers.append(nn.BatchNorm2d(32))
        layers.append(nn.ReLU())
        layers.append(nn.MaxPool2d(2, 2)) # 100 -> 50
        
        # Block 2: 32 -> 64
        layers.append(nn.Conv2d(32, 64, kernel_size=3, padding=1))
        if use_batchnorm: layers.append(nn.BatchNorm2d(64))
        layers.append(nn.ReLU())
        layers.append(nn.MaxPool2d(2, 2)) # 50 -> 25
        
        # Block 3: 64 -> 128
        layers.append(nn.Conv2d(64, 128, kernel_size=3, padding=1))
        if use_batchnorm: layers.append(nn.BatchNorm2d(128))
        layers.append(nn.ReLU())
        layers.append(nn.MaxPool2d(2, 2)) # 25 -> 12
        
        # Block 4: 128 -> 256 (Deep features for robustness)
        layers.append(nn.Conv2d(128, 256, kernel_size=3, padding=1))
        if use_batchnorm: layers.append(nn.BatchNorm2d(256))
        layers.append(nn.ReLU())
        layers.append(nn.MaxPool2d(2, 2)) # 12 -> 6
        
        self.features = nn.Sequential(*layers)
        
        # Classifier
        self.flatten_dim = 256 * 6 * 6
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, 1024),
            nn.ReLU(),
            # Dropout is added dynamically based on config
            nn.Dropout(0.5) if use_dropout else nn.Identity(), 
            nn.Linear(1024, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# ==========================================
# 4. The Training Engine (Exercise 7)
# ==========================================
def run_experiment(name, model, weight_decay=0.0):
    print(f"\n🧪 Experiment: {name}")
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)
    
    # LR Scheduler (Helpful for 30 epochs)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    
    stats = {'train_loss': [], 'val_acc': []}
    
    start_time = time.time()
    
    for epoch in range(EPOCHS):
        # Train
        model.train()
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # Validate
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                outputs = model(x)
                _, preds = torch.max(outputs, 1)
                total += y.size(0)
                correct += (preds == y).sum().item()
        
        val_acc = 100 * correct / total
        
        # Recording
        stats['train_loss'].append(train_loss / len(train_loader))
        stats['val_acc'].append(val_acc)
        scheduler.step(val_acc)
        
        if (epoch+1) % 5 == 0:
            print(f"   Epoch {epoch+1}/{EPOCHS} | Val Acc: {val_acc:.2f}% | Loss: {train_loss/len(train_loader):.4f}")

    print(f"   ⏱️ Time: {time.time() - start_time:.1f}s | Best Acc: {max(stats['val_acc']):.2f}%")
    return stats, model

# ==========================================
# 5. Experiment Comparison (Exercise 6c/8c)
# ==========================================

# Model 1: Baseline (Simple CNN, No Reg)
# Vulnerable to overfitting, might learn noise.
stats_base, _ = run_experiment(
    "Baseline (No Reg)", 
    FruitCNN(use_dropout=False, use_batchnorm=False), 
    weight_decay=0.0
)

# Model 2: L2 Regularization (Weight Decay)
# Penalizes large weights to force smoother decision boundaries.
stats_l2, _ = run_experiment(
    "L2 Regularization (1e-4)", 
    FruitCNN(use_dropout=False, use_batchnorm=False), 
    weight_decay=1e-4
)

# Model 3: Robust Net (Dropout + Batch Norm)
# Batch Norm stabilizes deep nets. Dropout forces redundancy.
# This is usually the winner for generalization.
stats_robust, best_model = run_experiment(
    "Robust (Dropout+BN)", 
    FruitCNN(use_dropout=True, use_batchnorm=True), 
    weight_decay=0.0
)

# ==========================================
# 6. Visualization
# ==========================================
plt.figure(figsize=(10, 6))
plt.plot(stats_base['val_acc'], label='Baseline', linestyle='--')
plt.plot(stats_l2['val_acc'], label='L2 Regularization')
plt.plot(stats_robust['val_acc'], label='Robust (Dropout+BN)', linewidth=2.5)
plt.title(f"Robustness Comparison: {EPOCHS} Epochs on {DEVICE}")
plt.xlabel("Epochs")
plt.ylabel("Validation Accuracy (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# ==========================================
# 7. Final Predictions (For Unlabelled Test)
# ==========================================
class UnlabelledDataset(Dataset):
    def __init__(self, root_dir, transform):
        self.root_dir = root_dir
        self.transform = transform
        self.images = [f for f in os.listdir(root_dir) if f.lower().endswith('.jpg')]
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.root_dir, img_name)
        img = Image.open(img_path).convert('RGB')
        return self.transform(img), img_name

# Fix for Windows multiprocessing bug in Notebooks
test_loader = DataLoader(
    UnlabelledDataset(TEST_DIR, clean_transforms), 
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0 
)

print(f"\n🔮 Generating Predictions using 'Robust' model...")
best_model.eval()
predictions = []

with torch.no_grad():
    for x, filenames in test_loader:
        x = x.to(DEVICE)
        outputs = best_model(x)
        _, preds = torch.max(outputs, 1)
        for i in range(len(filenames)):
            predictions.append([filenames[i], class_names[preds[i]]])

import csv
with open('submission.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['filename', 'label'])
    writer.writerows(predictions)

print("✅ submission.csv created!")

🚀 Project Running on: cuda
⚙️  Config: 8 Workers | Batch Size 128
✅ Data Prepared: 13483 Train | 3371 Val

🧪 Experiment: Baseline (No Reg)
   Epoch 5/15 | Val Acc: 100.00% | Loss: 0.0004
   Epoch 10/15 | Val Acc: 100.00% | Loss: 0.0000


KeyboardInterrupt: 